<a href="https://colab.research.google.com/github/lloydakresi/ml_journey/blob/main/Seq2Seq_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
%cd drive/MyDrive

/content/drive/MyDrive


In [ ]:
%ls

In [5]:
dataset_path = "fra.txt"

Remove excess text (attributions) and non-break space chars

In [6]:
def preprocess_text(text):
  text = text.split("CC-BY")[0].strip()
  text = text.replace("\u202f", " ").replace("\xa0", " ")
  no_space = lambda char, prev_char: char in ".,?!" and prev_char != " "
  out = [" " + char if i > 0 and no_space(char, text[i-1]) else char
         for i, char in enumerate(text.lower())]
  result = "".join(out)
  return(result)

Word-level tokenization

In [7]:
def tokenize_data(text, src, tgt):
  parts = text.split("\t")
  if len(parts) == 2:
    src.append([t for t in f"{parts[0]}".split(" ") if t])
    tgt.append([t for t in f"{parts[1]}".split(" ") if t])

In [57]:
x = torch.tensor([True, False, False, True]).type(torch.int32).sum(0)
x

tensor(2)

In [8]:
src = []
tgt = []

Preprocess and tokenize the data

In [9]:
with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

Pad or truncate the text to obtain a regular shape for computationally efficient operations

In [10]:
def pad_or_truncate(text, num_steps=10):
  if len(text) >= num_steps:
    text = text[:num_steps-1] + ["<eos>"]
  else:
    text += ["<eos>"]
    text += ["<pad>"] * (num_steps - len(text))
  return text


Lookup table

In [11]:
from collections import Counter

def Vocab(src=src, min_freq=2):

  def flatten(src=src):
    return [item for text in src for item in text]

  flattened_seq = flatten()
  counter = Counter(flattened_seq)
  vocab = {"<unk>":0, "<bos>":1}
  i = 2

  for token, freq in counter.items():
    if freq > min_freq and token not in vocab.keys():
      vocab[token] = i
      i += 1
  return vocab

def encode(vocab, word):
  return vocab.get(word, vocab["<unk>"])



In [20]:
tgt[210000]

['<bos>',
 'ça',
 'ne',
 'signifie',
 'pas',
 'que',
 'tu',
 'ne',
 'devrais',
 'pas',
 '<eos>']

In [12]:
src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
tgt = [["<bos>"] + t for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

In [21]:
lookup_fxn = lambda sentence, vocab: [encode(vocab, s) for s in sentence]
src_lookup = [lookup_fxn(s, eng_vocab) for s in src]
tgt_lookup = [lookup_fxn(t, fr_vocab) for t in tgt]


In [22]:
src_lookup = torch.tensor(src_lookup, dtype=torch.int32)
tgt_lookup = torch.tensor(tgt_lookup, dtype=torch.int32)

</bos/> is for input fed to the decoder. serves as a signal to the decoder to start producing an output. </eos/> is added to the label as a signal to the decoder to stop outputing tokens.

In [23]:
label = tgt_lookup[:, 1:]
decoder_input = tgt_lookup[:, :-1]

label_valid_len = (label != fr_vocab["<pad>"]).type(torch.int32).sum(1)
decoder_valid_len = (decoder_input != fr_vocab["<pad>"]).type(torch.int32).sum(1)
src_valid_len = (src_lookup != eng_vocab["<pad>"]).type(torch.int32).sum(1)


In [24]:
class Embedding():
  def __init__(self, vocab_size, feature_size):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.embedding = torch.randn((vocab_size, feature_size))

  def __call__(self, idx):
    return self.embedding[idx]

  def parameters(self):
    return [self.embedding]

  def __repr__(self):
    return f"Embedding({self.embedding.shape})"

In [25]:
dense = Embedding(10, 20)
idx = torch.arange(0, 10).view(2, 5)
dense(idx).shape

torch.Size([2, 5, 20])

In [26]:
from sequential_models import Linear, BasicRNN

In [27]:
class Encoder():
  def __init__(self,
               vocab_size, #number of unique tokens
               feature_size, #number of embedding features
               n_neurons, #number of neurons in the RNN layers
               output_dim #number of neurons in the final Linear layer
               ):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons
    self.context_dim = output_dim

    self.embedding = Embedding(vocab_size, feature_size)
    self.rnn1 = BasicRNN(feature_size, n_neurons)
    self.rnn2 = BasicRNN(n_neurons, n_neurons)


  def __call__(self, x):
    dense_input = self.embedding[x]
    output1, h1 = self.rnn1(dense_input)
    output, h2 = self.rnn2(output1)
    return (h1, h2)



  def __repr__(self):
    rep = f"Encoder(Embedding={self.embedding.embedding.shape},\nRNN=({self.feature_size, self.neurons}), \nRNN=({self.neurons, self.neurons}))"
    return rep

  def parameters(self):
    params = [self.embedding.parameters() + self.rnn1.parameters() + self.rnn2.parameters()]
    return params

In [ ]:
class Decoder():
  def __init__(self, vocab_size, feature_size, n_neurons):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons

    self.embedding = Embedding(vocab_size, feature_size)
    self.rnn1 = BasicRNN(feature_size, n_neurons)
    self.rnn1.W_x = torch.randn((feature_size + n_neurons, n_neurons))
    self.rnn2 = BasicRNN(n_neurons, n_neurons)
    self.linear = Linear(n_neurons, vocab_size)



  def __call__(self, x, context):
    embedded = self.embedding(x)

    context = context.tile((embedded.shape[0], 1, 1))
    input = torch.cat((embedded, context), -1)

    output1, h1 = self.rnn1(input)
    output2, h2 = self.rnn2(output1)
    final_outputs = linear(output2)

    return final_outputs


  def __repr__(self):
    rep = f"Decoder(Embedding={self.embedding.embedding.shape},\nRNN=({self.feature_size, self.neurons}), \nRNN=({self.neurons, self.neurons}))"
    return rep


  def parameters(self):
    params = [self.embedding.parameters() + self.rnn1.parameters() + self.rnn2.parameters()]
    return params
